# Final Project Scope

The project uses the Walmart M5 dataset only.

The core spine is demand heterogeneity. Item-store series will be classified into smooth, erratic, intermittent, and lumpy demand using ADI and CV².

The models are:
1. Seasonal naive baseline
2. XGBoost as the primary machine learning model
3. A limited LSTM feasibility study

WRMSSE is the primary evaluation metric. MAE is used only as a secondary metric for interpretation.

Validation will use walk-forward TimeSeriesSplit with 5 folds and 28-day test windows. Random splitting will not be used.

From Step 13 onward, every result will be reported by demand type.

## Conditional Extension Note

NOAA weather data is not part of the active implementation yet. It will only be considered after Step 12 if the supervisor approves it and the data can be merged cleanly.

Google Trends is removed from the project and will not be used.

# Step 9: Reshape Sales Data from Wide to Long Format

The raw M5 sales file is stored in wide format. Each row represents one item-store series, and daily sales are stored across 1,941 day columns.

For modelling and merging with calendar and price data, I reshape the data into long format. In the long version, each row represents one item-store-day observation.

Expected long-format size:

30,490 item-store series × 1,941 days = 59,181,090 rows.

To manage memory, I reshape and save the data store by store. This creates 10 files, one for each Walmart store.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

np.random.seed(42)

raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")
sales_long_store_path = processed_data_path / "sales_long_by_store"

processed_data_path.mkdir(parents=True, exist_ok=True)
sales_long_store_path.mkdir(parents=True, exist_ok=True)

print("Processed data path:", processed_data_path)
print("Store-level output path:", sales_long_store_path)

Processed data path: ..\data\processed
Store-level output path: ..\data\processed\sales_long_by_store


In [17]:
 sales = pd.read_csv(raw_data_path / "sales_train_evaluation.csv")

print("Sales shape:", sales.shape)
sales.head()

Sales shape: (30490, 1947)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [18]:
id_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
sales_day_columns = [col for col in sales.columns if col.startswith("d_")]

print("Number of item-store series:", sales.shape[0])
print("Number of sales day columns:", len(sales_day_columns))
print("First sales day:", sales_day_columns[0])
print("Last sales day:", sales_day_columns[-1])

expected_total_rows = sales.shape[0] * len(sales_day_columns)

print("Expected long-format rows:", expected_total_rows)

Number of item-store series: 30490
Number of sales day columns: 1941
First sales day: d_1
Last sales day: d_1941
Expected long-format rows: 59181090


In [21]:
def count_csv_data_rows(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        row_count = sum(1 for _ in file) - 1
    return row_count

In [22]:
store_ids = list(sales["store_id"].unique())

expected_store_rows = (
    sales.groupby("store_id")
    .size()
    .reset_index(name="wide_rows")
)

expected_store_rows["expected_long_rows"] = (
    expected_store_rows["wide_rows"] * len(sales_day_columns)
)

expected_store_rows

,store_id,wide_rows,expected_long_rows
0,CA_1,3049,5918109
1,CA_2,3049,5918109
2,CA_3,3049,5918109
3,CA_4,3049,5918109
4,TX_1,3049,5918109
5,TX_2,3049,5918109
6,TX_3,3049,5918109
7,WI_1,3049,5918109
8,WI_2,3049,5918109
9,WI_3,3049,5918109


In [23]:
store_row_records = []

for store in store_ids:
    output_file = sales_long_store_path / f"{store}_sales_long.csv"
    
    expected_rows_for_store = int(
        expected_store_rows.loc[
            expected_store_rows["store_id"] == store,
            "expected_long_rows"
        ].iloc[0]
    )
    
    recreate_file = True
    
    if output_file.exists():
        existing_rows = count_csv_data_rows(output_file)
        
        if existing_rows == expected_rows_for_store:
            print(f"{store}: existing file valid with {existing_rows:,} rows. Skipping.")
            recreate_file = False
            
            store_row_records.append({
                "store_id": store,
                "expected_rows": expected_rows_for_store,
                "created_rows": existing_rows,
                "status": "existing_valid",
                "output_file": str(output_file)
            })
        else:
            print(
                f"{store}: existing file has {existing_rows:,} rows, "
                f"expected {expected_rows_for_store:,}. Recreating."
            )
    
    if recreate_file:
        print(f"{store}: reshaping now.")
        
        store_sales = sales.loc[
            sales["store_id"] == store,
            id_columns + sales_day_columns
        ].copy()
        
        store_long = store_sales.melt(
            id_vars=id_columns,
            value_vars=sales_day_columns,
            var_name="d",
            value_name="sales"
        )
        
        store_long["sales"] = store_long["sales"].astype("int32")
        
        output_file = sales_long_store_path / f"{store}_sales_long.csv"
        store_long.to_csv(output_file, index=False)
        
        created_rows = count_csv_data_rows(output_file)
        
        print(f"{store}: saved {created_rows:,} rows.")
        
        store_row_records.append({
            "store_id": store,
            "expected_rows": expected_rows_for_store,
            "created_rows": created_rows,
            "status": "created_or_recreated",
            "output_file": str(output_file)
        })
        
        del store_sales
        del store_long
        gc.collect()

store_row_counts = pd.DataFrame(store_row_records)
store_row_counts

CA_1: existing file valid with 5,918,109 rows. Skipping.
CA_2: reshaping now.
CA_2: saved 5,918,109 rows.
CA_3: reshaping now.
CA_3: saved 5,918,109 rows.
CA_4: reshaping now.
CA_4: saved 5,918,109 rows.
TX_1: reshaping now.
TX_1: saved 5,918,109 rows.
TX_2: reshaping now.
TX_2: saved 5,918,109 rows.
TX_3: reshaping now.
TX_3: saved 5,918,109 rows.
WI_1: reshaping now.
WI_1: saved 5,918,109 rows.
WI_2: reshaping now.
WI_2: saved 5,918,109 rows.
WI_3: reshaping now.
WI_3: saved 5,918,109 rows.


,store_id,expected_rows,created_rows,status,output_file
0,CA_1,5918109,5918109,existing_valid,..\data\processed\sales_long_by_store\CA_1_sal...
1,CA_2,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\CA_2_sal...
2,CA_3,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\CA_3_sal...
3,CA_4,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\CA_4_sal...
4,TX_1,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\TX_1_sal...
5,TX_2,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\TX_2_sal...
6,TX_3,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\TX_3_sal...
7,WI_1,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\WI_1_sal...
8,WI_2,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\WI_2_sal...
9,WI_3,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\WI_3_sal...


In [24]:
total_created_rows = store_row_counts["created_rows"].sum()
all_store_files_valid = (
    store_row_counts["expected_rows"] == store_row_counts["created_rows"]
).all()

print("Expected total rows:", expected_total_rows)
print("Created total rows:", total_created_rows)
print("Total row match:", expected_total_rows == total_created_rows)
print("All store files valid:", all_store_files_valid)

store_row_counts

Expected total rows: 59181090
Created total rows: 59181090
Total row match: True
All store files valid: True


,store_id,expected_rows,created_rows,status,output_file
0,CA_1,5918109,5918109,existing_valid,..\data\processed\sales_long_by_store\CA_1_sal...
1,CA_2,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\CA_2_sal...
2,CA_3,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\CA_3_sal...
3,CA_4,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\CA_4_sal...
4,TX_1,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\TX_1_sal...
5,TX_2,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\TX_2_sal...
6,TX_3,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\TX_3_sal...
7,WI_1,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\WI_1_sal...
8,WI_2,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\WI_2_sal...
9,WI_3,5918109,5918109,created_or_recreated,..\data\processed\sales_long_by_store\WI_3_sal...


In [25]:
store_row_counts.to_csv(
    processed_data_path / "sales_long_store_row_counts.csv",
    index=False
)

print("Saved:", processed_data_path / "sales_long_store_row_counts.csv")

Saved: ..\data\processed\sales_long_store_row_counts.csv


In [27]:
check_file = sales_long_store_path / "CA_1_sales_long.csv"

check_sample = pd.read_csv(check_file, nrows=10)

print("CA_1 sample shape:", check_sample.shape)
check_sample

CA_1 sample shape: (10, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
5,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
6,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
7,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12
8,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,d_1,2
9,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


## Step 9 Interpretation

The sales data was reshaped from wide format to long format. The original sales file contained 30,490 item-store series and 1,941 daily sales columns, giving an expected long-format size of 59,181,090 item-store-day rows.

To manage memory, I reshaped the data store by store and saved 10 separate files. Each store file contains 5,918,109 rows, calculated from 3,049 items multiplied by 1,941 sales days.

The final validation confirms that the 10 store-level files contain 59,181,090 rows in total. This matches the expected row count, so the wide-to-long reshape is complete.

This long-format structure is required for Step 10, where sales will be merged with calendar and weekly price data.

### Surprise Log

The main practical issue in Step 9 was the size of the reshaped data. The long-format sales data contains 59,181,090 rows, so saving one full file at once would be risky. Splitting the reshape into 10 store-level files kept each output to 5,918,109 rows and made the validation easier.

# Step 10: Merge Sales, Calendar, and Price Data

In this step, I merge the long-format sales data with calendar and weekly price information.

The sales data is merged with the calendar file using the `d` column. The result is then merged with the price file using `store_id`, `item_id`, and `wm_yr_wk`.

The merge is completed store by store to manage memory. Zero-sales rows are kept because sparsity is central to the dissertation.

In [29]:
calendar = pd.read_csv(raw_data_path / "calendar.csv")
prices = pd.read_csv(raw_data_path / "sell_prices.csv")

print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)

print("Calendar date range:", calendar["date"].min(), "to", calendar["date"].max())
print("Calendar d range:", calendar["d"].min(), "to", calendar["d"].max())

calendar.head()

Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Calendar date range: 2011-01-29 to 2016-06-19
Calendar d range: d_1 to d_999


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [30]:
print("Calendar duplicate d values:", calendar["d"].duplicated().sum())

price_duplicate_keys = prices.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print("Duplicate price keys:", price_duplicate_keys)

Calendar duplicate d values: 0
Duplicate price keys: 0


In [31]:
calendar_columns = [
    "d",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

calendar_small = calendar[calendar_columns].copy()

calendar_small["date"] = pd.to_datetime(calendar_small["date"])

calendar_small.head()

,d,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,d_1,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,d_2,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,d_3,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,d_4,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1,1,0
4,d_5,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1,0,1


In [32]:
prices_small = prices.copy()

prices_small["sell_price"] = prices_small["sell_price"].astype("float32")

prices_small.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [33]:
merged_store_path = processed_data_path / "merged_by_store"
merged_store_path.mkdir(parents=True, exist_ok=True)

print("Merged output folder:", merged_store_path)

Merged output folder: ..\data\processed\merged_by_store


In [34]:
test_store = "CA_1"

sales_file = sales_long_store_path / f"{test_store}_sales_long.csv"

sales_store = pd.read_csv(sales_file)

print("Sales store shape before merge:", sales_store.shape)
sales_store.head()

Sales store shape before merge: (5918109, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [35]:
merged_store = sales_store.merge(
    calendar_small,
    on="d",
    how="left",
    validate="many_to_one"
)

print("Shape after calendar merge:", merged_store.shape)
print("Missing calendar dates:", merged_store["date"].isna().sum())

Shape after calendar merge: (5918109, 21)
Missing calendar dates: 0


In [36]:
def assign_snap_flag(row):
    if row["state_id"] == "CA":
        return row["snap_CA"]
    elif row["state_id"] == "TX":
        return row["snap_TX"]
    elif row["state_id"] == "WI":
        return row["snap_WI"]
    else:
        return 0

merged_store["snap"] = merged_store.apply(assign_snap_flag, axis=1)

In [37]:
state = merged_store["state_id"].iloc[0]

if state == "CA":
    merged_store["snap"] = merged_store["snap_CA"]
elif state == "TX":
    merged_store["snap"] = merged_store["snap_TX"]
elif state == "WI":
    merged_store["snap"] = merged_store["snap_WI"]

merged_store = merged_store.drop(columns=["snap_CA", "snap_TX", "snap_WI"])

In [38]:
merged_store = merged_store.merge(
    prices_small,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

print("Shape after price merge:", merged_store.shape)

price_missing_before_ffill = merged_store["sell_price"].isna().sum()
price_missing_before_ffill_pct = price_missing_before_ffill / len(merged_store) * 100

print("Missing sell_price before forward fill:", price_missing_before_ffill)
print("Missing sell_price before forward fill %:", round(price_missing_before_ffill_pct, 4))

Shape after price merge: (5918109, 20)
Missing sell_price before forward fill: 1129842
Missing sell_price before forward fill %: 19.0913


In [39]:
merged_store = merged_store.sort_values(["id", "date"])

merged_store["sell_price"] = (
    merged_store
    .groupby("id")["sell_price"]
    .ffill()
)

price_missing_after_ffill = merged_store["sell_price"].isna().sum()
price_missing_after_ffill_pct = price_missing_after_ffill / len(merged_store) * 100

print("Missing sell_price after forward fill:", price_missing_after_ffill)
print("Missing sell_price after forward fill %:", round(price_missing_after_ffill_pct, 4))

Missing sell_price after forward fill: 1129842
Missing sell_price after forward fill %: 19.0913


In [40]:
print("Store:", test_store)
print("Rows after merge:", len(merged_store))
print("Unique dates:", merged_store["date"].nunique())
print("Date range:", merged_store["date"].min(), "to", merged_store["date"].max())
print("Total sales after merge:", merged_store["sales"].sum())

Store: CA_1
Rows after merge: 5918109
Unique dates: 1941
Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00
Total sales after merge: 7832248


In [41]:
original_ca1_total_sales = (
    sales.loc[sales["store_id"] == test_store, sales_day_columns]
    .sum()
    .sum()
)

merged_ca1_total_sales = merged_store["sales"].sum()

print("Original CA_1 total sales:", original_ca1_total_sales)
print("Merged CA_1 total sales:", merged_ca1_total_sales)
print("Sales total match:", original_ca1_total_sales == merged_ca1_total_sales)

Original CA_1 total sales: 7832248
Merged CA_1 total sales: 7832248
Sales total match: True


In [42]:
try:
    import pyarrow
    print("pyarrow is available.")
except ModuleNotFoundError:
    print("pyarrow is not installed. Install it with: pip install pyarrow")

pyarrow is available.


In [43]:
test_output_file = merged_store_path / f"{test_store}_merged.parquet"

merged_store.to_parquet(test_output_file, index=False)

print("Saved:", test_output_file)

Saved: ..\data\processed\merged_by_store\CA_1_merged.parquet


In [44]:
del sales_store
del merged_store
gc.collect()

0

In [45]:
merge_validation_records = []

for store in store_ids:
    print(f"Processing store: {store}")
    
    sales_file = sales_long_store_path / f"{store}_sales_long.csv"
    output_file = merged_store_path / f"{store}_merged.parquet"
    
    sales_store = pd.read_csv(sales_file)
    
    rows_before = len(sales_store)
    sales_before = sales_store["sales"].sum()
    
    merged_store = sales_store.merge(
        calendar_small,
        on="d",
        how="left",
        validate="many_to_one"
    )
    
    missing_calendar_rows = merged_store["date"].isna().sum()
    
    state = merged_store["state_id"].iloc[0]
    
    if state == "CA":
        merged_store["snap"] = merged_store["snap_CA"]
    elif state == "TX":
        merged_store["snap"] = merged_store["snap_TX"]
    elif state == "WI":
        merged_store["snap"] = merged_store["snap_WI"]
    
    merged_store = merged_store.drop(columns=["snap_CA", "snap_TX", "snap_WI"])
    
    merged_store = merged_store.merge(
        prices_small,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one"
    )
    
    price_missing_before_ffill = merged_store["sell_price"].isna().sum()
    price_missing_before_ffill_pct = price_missing_before_ffill / len(merged_store) * 100
    
    merged_store = merged_store.sort_values(["id", "date"])
    
    merged_store["sell_price"] = (
        merged_store
        .groupby("id")["sell_price"]
        .ffill()
    )
    
    price_missing_after_ffill = merged_store["sell_price"].isna().sum()
    price_missing_after_ffill_pct = price_missing_after_ffill / len(merged_store) * 100
    
    rows_after = len(merged_store)
    sales_after = merged_store["sales"].sum()
    
    date_min = merged_store["date"].min()
    date_max = merged_store["date"].max()
    
    merged_store.to_parquet(output_file, index=False)
    
    merge_validation_records.append({
        "store_id": store,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "row_count_match": rows_before == rows_after,
        "sales_before": sales_before,
        "sales_after": sales_after,
        "sales_total_match": sales_before == sales_after,
        "missing_calendar_rows": missing_calendar_rows,
        "price_missing_before_ffill": price_missing_before_ffill,
        "price_missing_before_ffill_pct": round(price_missing_before_ffill_pct, 4),
        "price_missing_after_ffill": price_missing_after_ffill,
        "price_missing_after_ffill_pct": round(price_missing_after_ffill_pct, 4),
        "date_min": date_min,
        "date_max": date_max,
        "output_file": str(output_file)
    })
    
    print(f"{store}: saved {rows_after:,} rows.")
    print(f"{store}: missing calendar rows = {missing_calendar_rows}")
    print(f"{store}: price missing after ffill = {price_missing_after_ffill} ({round(price_missing_after_ffill_pct, 4)}%)")
    
    del sales_store
    del merged_store
    gc.collect()

merge_validation = pd.DataFrame(merge_validation_records)
merge_validation

Processing store: CA_1
CA_1: saved 5,918,109 rows.
CA_1: missing calendar rows = 0
CA_1: price missing after ffill = 1129842 (19.0913%)
Processing store: CA_2
CA_2: saved 5,918,109 rows.
CA_2: missing calendar rows = 0
CA_2: price missing after ffill = 1556961 (26.3084%)
Processing store: CA_3
CA_3: saved 5,918,109 rows.
CA_3: missing calendar rows = 0
CA_3: price missing after ffill = 1160796 (19.6143%)
Processing store: CA_4
CA_4: saved 5,918,109 rows.
CA_4: missing calendar rows = 0
CA_4: price missing after ffill = 1265551 (21.3844%)
Processing store: TX_1
TX_1: saved 5,918,109 rows.
TX_1: missing calendar rows = 0
TX_1: price missing after ffill = 1120154 (18.9276%)
Processing store: TX_2
TX_2: saved 5,918,109 rows.
TX_2: missing calendar rows = 0
TX_2: price missing after ffill = 1110228 (18.7598%)
Processing store: TX_3
TX_3: saved 5,918,109 rows.
TX_3: missing calendar rows = 0
TX_3: price missing after ffill = 1180942 (19.9547%)
Processing store: WI_1
WI_1: saved 5,918,109 row

,store_id,rows_before,rows_after,row_count_match,sales_before,sales_after,sales_total_match,missing_calendar_rows,price_missing_before_ffill,price_missing_before_ffill_pct,price_missing_after_ffill,price_missing_after_ffill_pct,date_min,date_max,output_file
0,CA_1,5918109,5918109,True,7832248,7832248,True,0,1129842,19.0913,1129842,19.0913,2011-01-29,2016-05-22,..\data\processed\merged_by_store\CA_1_merged....
1,CA_2,5918109,5918109,True,5818395,5818395,True,0,1556961,26.3084,1556961,26.3084,2011-01-29,2016-05-22,..\data\processed\merged_by_store\CA_2_merged....
2,CA_3,5918109,5918109,True,11363540,11363540,True,0,1160796,19.6143,1160796,19.6143,2011-01-29,2016-05-22,..\data\processed\merged_by_store\CA_3_merged....
3,CA_4,5918109,5918109,True,4182534,4182534,True,0,1265551,21.3844,1265551,21.3844,2011-01-29,2016-05-22,..\data\processed\merged_by_store\CA_4_merged....
4,TX_1,5918109,5918109,True,5692823,5692823,True,0,1120154,18.9276,1120154,18.9276,2011-01-29,2016-05-22,..\data\processed\merged_by_store\TX_1_merged....
5,TX_2,5918109,5918109,True,7329642,7329642,True,0,1110228,18.7598,1110228,18.7598,2011-01-29,2016-05-22,..\data\processed\merged_by_store\TX_2_merged....
6,TX_3,5918109,5918109,True,6205940,6205940,True,0,1180942,19.9547,1180942,19.9547,2011-01-29,2016-05-22,..\data\processed\merged_by_store\TX_3_merged....
7,WI_1,5918109,5918109,True,5261506,5261506,True,0,1357342,22.9354,1357342,22.9354,2011-01-29,2016-05-22,..\data\processed\merged_by_store\WI_1_merged....
8,WI_2,5918109,5918109,True,6697988,6697988,True,0,1271529,21.4854,1271529,21.4854,2011-01-29,2016-05-22,..\data\processed\merged_by_store\WI_2_merged....
9,WI_3,5918109,5918109,True,6542557,6542557,True,0,1146068,19.3654,1146068,19.3654,2011-01-29,2016-05-22,..\data\processed\merged_by_store\WI_3_merged....


In [46]:
total_rows_before = merge_validation["rows_before"].sum()
total_rows_after = merge_validation["rows_after"].sum()

total_missing_calendar = merge_validation["missing_calendar_rows"].sum()

total_price_missing_before = merge_validation["price_missing_before_ffill"].sum()
total_price_missing_after = merge_validation["price_missing_after_ffill"].sum()

total_price_missing_before_pct = total_price_missing_before / total_rows_after * 100
total_price_missing_after_pct = total_price_missing_after / total_rows_after * 100

print("Total rows before merge:", total_rows_before)
print("Total rows after merge:", total_rows_after)
print("Row count preserved:", total_rows_before == total_rows_after)

print("Total missing calendar rows:", total_missing_calendar)

print("Total missing prices before forward fill:", total_price_missing_before)
print("Total missing prices before forward fill %:", round(total_price_missing_before_pct, 4))

print("Total missing prices after forward fill:", total_price_missing_after)
print("Total missing prices after forward fill %:", round(total_price_missing_after_pct, 4))

print("Minimum date:", merge_validation["date_min"].min())
print("Maximum date:", merge_validation["date_max"].max())

print("All store row counts valid:", merge_validation["row_count_match"].all())
print("All store sales totals valid:", merge_validation["sales_total_match"].all())

Total rows before merge: 59181090
Total rows after merge: 59181090
Row count preserved: True
Total missing calendar rows: 0
Total missing prices before forward fill: 12299413
Total missing prices before forward fill %: 20.7827
Total missing prices after forward fill: 12299413
Total missing prices after forward fill %: 20.7827
Minimum date: 2011-01-29 00:00:00
Maximum date: 2016-05-22 00:00:00
All store row counts valid: True
All store sales totals valid: True


In [47]:
merge_validation.to_csv(
    processed_data_path / "step10_merge_validation.csv",
    index=False
)

print("Saved:", processed_data_path / "step10_merge_validation.csv")

Saved: ..\data\processed\step10_merge_validation.csv


In [48]:
combined_output_file = processed_data_path / "m5_merged_base.parquet"

merged_parts = []

for store in store_ids:
    file_path = merged_store_path / f"{store}_merged.parquet"
    part = pd.read_parquet(file_path)
    merged_parts.append(part)

m5_merged_base = pd.concat(merged_parts, ignore_index=True)

print("Combined shape:", m5_merged_base.shape)

m5_merged_base.to_parquet(combined_output_file, index=False)

print("Saved combined file:", combined_output_file)

del merged_parts
del m5_merged_base
gc.collect()

Combined shape: (59181090, 20)
Saved combined file: ..\data\processed\m5_merged_base.parquet


0

In [49]:
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

combined_output_file = processed_data_path / "m5_merged_base.parquet"

store_parquet_files = [
    merged_store_path / f"{store}_merged.parquet"
    for store in store_ids
]

writer = None
total_written_rows = 0

for file_path in store_parquet_files:
    print(f"Adding: {file_path.name}")
    
    table = pq.read_table(file_path)
    
    if writer is None:
        writer = pq.ParquetWriter(combined_output_file, table.schema)
    
    writer.write_table(table)
    total_written_rows += table.num_rows
    
    print(f"Rows added: {table.num_rows:,}")

if writer is not None:
    writer.close()

print("Saved combined parquet:", combined_output_file)
print("Total written rows:", total_written_rows)

Adding: CA_1_merged.parquet
Rows added: 5,918,109
Adding: CA_2_merged.parquet
Rows added: 5,918,109
Adding: CA_3_merged.parquet
Rows added: 5,918,109
Adding: CA_4_merged.parquet
Rows added: 5,918,109
Adding: TX_1_merged.parquet
Rows added: 5,918,109
Adding: TX_2_merged.parquet
Rows added: 5,918,109
Adding: TX_3_merged.parquet
Rows added: 5,918,109
Adding: WI_1_merged.parquet
Rows added: 5,918,109
Adding: WI_2_merged.parquet
Rows added: 5,918,109
Adding: WI_3_merged.parquet
Rows added: 5,918,109
Saved combined parquet: ..\data\processed\m5_merged_base.parquet
Total written rows: 59181090


In [50]:
combined_parquet = pq.ParquetFile(combined_output_file)

print("Combined parquet rows:", combined_parquet.metadata.num_rows)
print("Expected rows:", 59181090)
print("Combined row match:", combined_parquet.metadata.num_rows == 59181090)

Combined parquet rows: 59181090
Expected rows: 59181090
Combined row match: True


## Step 10 Interpretation

The long-format sales data was merged with calendar and weekly price information. The calendar merge used the `d` column, while the price merge used `store_id`, `item_id`, and `wm_yr_wk`.

The merge preserved the full sales row count. The total number of rows before the merge was 59,181,090, and the total number of rows after the merge was also 59,181,090. The row count match was True.

There were 0 missing calendar rows after the merge. This confirms that every observed sales day matched correctly to the calendar file.

The observed sales date range after the merge was 2011-01-29 to 2016-05-22. This range ends at `d_1941`, which is the final observed sales day in the `sales_train_evaluation` file.

Price missingness before forward fill was 12,299,413 rows, equal to 20.7827%. After forward filling within each item-store series, missing price values remained at 12,299,413 rows, equal to 20.7827%. This means the missing prices are mainly before the first available price for some item-store series, so forward fill cannot fill them without using future information.

Zero-sales rows were kept because sparsity is central to the dissertation.


### Surprise Log

The main surprise in Step 10 was that forward filling did not reduce price missingness. Missing prices remained at 12,299,413 rows, or 20.7827%, after forward fill. This suggests that the missing values are mostly before the first recorded price for some item-store series, so they will need careful handling during feature engineering.


# Step 11: ADI and CV² Demand Classification

This step classifies all 30,490 item-store series into smooth, erratic, intermittent, and lumpy demand.

Average Demand Interval measures how frequently non-zero demand occurs. Squared Coefficient of Variation measures how variable the size of non-zero demand is.

The classification thresholds are:

- Smooth: ADI ≤ 1.32 and CV² ≤ 0.49
- Erratic: ADI ≤ 1.32 and CV² > 0.49
- Intermittent: ADI > 1.32 and CV² ≤ 0.49
- Lumpy: ADI > 1.32 and CV² > 0.49

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

np.random.seed(42)

raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")

sales = pd.read_csv(
    raw_data_path / "sales_train_evaluation.csv"
)

id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

sales_day_columns = [
    column for column in sales.columns
    if column.startswith("d_")
]

print("Sales shape:", sales.shape)
print("Item-store series:", len(sales))
print("Sales days:", len(sales_day_columns))

Sales shape: (30490, 1947)
Item-store series: 30490
Sales days: 1941


In [2]:
sales_values = sales[sales_day_columns].to_numpy(
    dtype=np.float32
)

total_periods = len(sales_day_columns)

non_zero_mask = sales_values > 0
non_zero_days = non_zero_mask.sum(axis=1)

total_sales = sales_values.sum(axis=1)
zero_days = total_periods - non_zero_days
zero_rate = zero_days / total_periods * 100

# ADI = total days / non-zero days
adi = np.divide(
    total_periods,
    non_zero_days,
    out=np.full(
        non_zero_days.shape,
        np.nan,
        dtype=np.float32
    ),
    where=non_zero_days > 0
)

# Mean demand on non-zero days
non_zero_mean = np.divide(
    total_sales,
    non_zero_days,
    out=np.full(
        total_sales.shape,
        np.nan,
        dtype=np.float32
    ),
    where=non_zero_days > 0
)

# Variance of demand on non-zero days
non_zero_square_sum = np.square(sales_values).sum(axis=1)

non_zero_variance = np.divide(
    non_zero_square_sum,
    non_zero_days,
    out=np.full(
        non_zero_square_sum.shape,
        np.nan,
        dtype=np.float32
    ),
    where=non_zero_days > 0
) - np.square(non_zero_mean)

# Remove tiny negative values caused by floating-point precision
non_zero_variance = np.maximum(non_zero_variance, 0)

# CV² = variance / mean²
cv2 = np.divide(
    non_zero_variance,
    np.square(non_zero_mean),
    out=np.full(
        non_zero_variance.shape,
        np.nan,
        dtype=np.float32
    ),
    where=non_zero_mean > 0
)

print("Finite ADI values:", np.isfinite(adi).sum())
print("Finite CV² values:", np.isfinite(cv2).sum())
print("Series with no non-zero sales:", (non_zero_days == 0).sum())

Finite ADI values: 30490
Finite CV² values: 30490
Series with no non-zero sales: 0


In [3]:
demand_classification = sales[id_columns].copy()

demand_classification["ADI"] = adi
demand_classification["CV2"] = cv2
demand_classification["zero_rate"] = zero_rate
demand_classification["total_sales"] = total_sales.astype("int64")
demand_classification["non_zero_days"] = non_zero_days.astype("int32")

demand_classification.head()

,id,item_id,dept_id,cat_id,store_id,state_id,ADI,CV2,zero_rate,total_sales,non_zero_days
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,4.451835,0.282900,77.537352,633,436
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,4.757353,0.233792,78.979907,500,408
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,8.475983,0.287932,88.201958,309,229
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,1.474924,0.583019,32.199897,3337,1316
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1.974568,0.400368,49.356002,1888,983


In [4]:
conditions = [
    (
        demand_classification["ADI"].le(1.32)
        & demand_classification["CV2"].le(0.49)
    ),
    (
        demand_classification["ADI"].le(1.32)
        & demand_classification["CV2"].gt(0.49)
    ),
    (
        demand_classification["ADI"].gt(1.32)
        & demand_classification["CV2"].le(0.49)
    ),
    (
        demand_classification["ADI"].gt(1.32)
        & demand_classification["CV2"].gt(0.49)
    )
]

choices = [
    "smooth",
    "erratic",
    "intermittent",
    "lumpy"
]

demand_classification["demand_type"] = np.select(
    conditions,
    choices,
    default="unclassified"
)

demand_classification["demand_type"].value_counts(
    dropna=False
)

demand_type
intermittent    23075
lumpy            5938
smooth            983
erratic           494
Name: count, dtype: int64

In [5]:
demand_type_order = [
    "smooth",
    "erratic",
    "intermittent",
    "lumpy",
    "unclassified"
]

demand_type_summary = (
    demand_classification
    .groupby("demand_type", observed=True)
    .agg(
        series_count=("id", "count"),
        mean_ADI=("ADI", "mean"),
        mean_CV2=("CV2", "mean"),
        mean_zero_rate=("zero_rate", "mean"),
        mean_total_sales=("total_sales", "mean"),
        mean_non_zero_days=("non_zero_days", "mean")
    )
    .reindex(demand_type_order)
    .dropna(how="all")
)

demand_type_summary["percentage"] = (
    demand_type_summary["series_count"]
    / len(demand_classification)
    * 100
)

demand_type_summary = demand_type_summary[
    [
        "series_count",
        "percentage",
        "mean_ADI",
        "mean_CV2",
        "mean_zero_rate",
        "mean_total_sales",
        "mean_non_zero_days"
    ]
].round(3)

demand_type_summary

,series_count,percentage,mean_ADI,mean_CV2,mean_zero_rate,mean_total_sales,mean_non_zero_days
demand_type,,,,,,,
smooth,983.0,3.224,1.168,0.361,13.798,13657.063,1673.177
erratic,494.0,1.620,1.205,0.724,16.686,13261.196,1617.119
intermittent,23075.0,75.681,7.259,0.287,73.458,1230.333,515.177
lumpy,5938.0,19.475,4.005,0.765,60.020,3125.854,776.015


In [6]:
summary_count = int(
    demand_type_summary["series_count"].sum()
)

unclassified_count = int(
    (
        demand_classification["demand_type"]
        == "unclassified"
    ).sum()
)

print("Total series:", len(demand_classification))
print("Summary count:", summary_count)
print(
    "Summary count match:",
    summary_count == len(demand_classification)
)

print(
    "Missing demand types:",
    demand_classification["demand_type"].isna().sum()
)

print("Unclassified series:", unclassified_count)

print(
    "Minimum ADI:",
    round(demand_classification["ADI"].min(), 3)
)

print(
    "Maximum ADI:",
    round(demand_classification["ADI"].max(), 3)
)

print(
    "Minimum CV²:",
    round(demand_classification["CV2"].min(), 3)
)

print(
    "Maximum CV²:",
    round(demand_classification["CV2"].max(), 3)
)

Total series: 30490
Summary count: 30490
Summary count match: True
Missing demand types: 0
Unclassified series: 0
Minimum ADI: 1.002
Maximum ADI: 161.75
Minimum CV²: 0.0
Maximum CV²: 106.745


In [7]:
category_by_demand_type = pd.crosstab(
    demand_classification["demand_type"],
    demand_classification["cat_id"],
    margins=True
)

category_by_demand_type

cat_id,FOODS,HOBBIES,HOUSEHOLD,All
demand_type,,,,
erratic,365,40,89,494
intermittent,9183,4523,9369,23075
lumpy,4108,1058,772,5938
smooth,714,29,240,983
All,14370,5650,10470,30490


In [8]:
category_by_demand_type_pct = (
    pd.crosstab(
        demand_classification["demand_type"],
        demand_classification["cat_id"],
        normalize="index"
    )
    * 100
).round(2)

category_by_demand_type_pct

cat_id,FOODS,HOBBIES,HOUSEHOLD
demand_type,,,
erratic,73.89,8.10,18.02
intermittent,39.80,19.60,40.60
lumpy,69.18,17.82,13.00
smooth,72.63,2.95,24.42


In [9]:
store_by_demand_type = pd.crosstab(
    demand_classification["demand_type"],
    demand_classification["store_id"],
    margins=True
)

store_by_demand_type

store_id,CA_1,CA_2,CA_3,CA_4,TX_1,TX_2,TX_3,WI_1,WI_2,WI_3,All
demand_type,,,,,,,,,,,
erratic,42,50,89,22,54,61,47,42,36,51,494
intermittent,2295,2184,2040,2548,2316,2186,2343,2476,2362,2325,23075
lumpy,578,742,695,413,590,695,566,470,588,601,5938
smooth,134,73,225,66,89,107,93,61,63,72,983
All,3049,3049,3049,3049,3049,3049,3049,3049,3049,3049,30490


In [10]:
store_by_demand_type_pct = (
    pd.crosstab(
        demand_classification["demand_type"],
        demand_classification["store_id"],
        normalize="index"
    )
    * 100
).round(2)

store_by_demand_type_pct

store_id,CA_1,CA_2,CA_3,CA_4,TX_1,TX_2,TX_3,WI_1,WI_2,WI_3
demand_type,,,,,,,,,,
erratic,8.50,10.12,18.02,4.45,10.93,12.35,9.51,8.50,7.29,10.32
intermittent,9.95,9.46,8.84,11.04,10.04,9.47,10.15,10.73,10.24,10.08
lumpy,9.73,12.50,11.70,6.96,9.94,11.70,9.53,7.92,9.90,10.12
smooth,13.63,7.43,22.89,6.71,9.05,10.89,9.46,6.21,6.41,7.32


In [11]:
weak_series = (
    demand_classification["non_zero_days"] < 10
)

weak_series_count = int(weak_series.sum())
weak_series_percentage = (
    weak_series_count
    / len(demand_classification)
    * 100
)

print(
    "Series with fewer than 10 non-zero days:",
    weak_series_count
)

print(
    "Percentage:",
    round(weak_series_percentage, 3)
)

Series with fewer than 10 non-zero days: 0
Percentage: 0.0


In [12]:
weak_series_by_type = (
    demand_classification
    .assign(weak_series=weak_series)
    .groupby("demand_type")
    .agg(
        weak_series_count=("weak_series", "sum"),
        total_series=("id", "count")
    )
)

weak_series_by_type["percentage"] = (
    weak_series_by_type["weak_series_count"]
    / weak_series_by_type["total_series"]
    * 100
).round(3)

weak_series_by_type

,weak_series_count,total_series,percentage
demand_type,,,
erratic,0,494,0.0
intermittent,0,23075,0.0
lumpy,0,5938,0.0
smooth,0,983,0.0


In [13]:
demand_classification.to_csv(
    processed_data_path / "demand_classification.csv",
    index=False
)

demand_type_summary.to_csv(
    processed_data_path / "demand_type_summary.csv"
)

category_by_demand_type.to_csv(
    processed_data_path / "category_by_demand_type.csv"
)

category_by_demand_type_pct.to_csv(
    processed_data_path / "category_by_demand_type_percentage.csv"
)

store_by_demand_type.to_csv(
    processed_data_path / "store_by_demand_type.csv"
)

store_by_demand_type_pct.to_csv(
    processed_data_path / "store_by_demand_type_percentage.csv"
)

weak_series_by_type.to_csv(
    processed_data_path / "weak_series_by_demand_type.csv"
)

print("Step 11 output files saved.")

Step 11 output files saved.


In [14]:
del sales_values
del non_zero_mask
del non_zero_square_sum

gc.collect()

0

In [15]:
# Recalculate ADI and CV² in float64 without loading the full array at once

number_of_series = len(sales)
chunk_size = 1000

adi_float64 = np.empty(number_of_series, dtype=np.float64)
cv2_float64 = np.empty(number_of_series, dtype=np.float64)

original_demand_types = (
    demand_classification["demand_type"]
    .copy()
    .reset_index(drop=True)
)

for start_row in range(0, number_of_series, chunk_size):
    end_row = min(start_row + chunk_size, number_of_series)
    
    chunk_values = (
        sales.iloc[start_row:end_row][sales_day_columns]
        .to_numpy(dtype=np.float64)
    )
    
    chunk_non_zero_mask = chunk_values > 0
    chunk_non_zero_days = chunk_non_zero_mask.sum(axis=1)
    
    chunk_total_sales = chunk_values.sum(axis=1)
    chunk_square_sum = np.square(chunk_values).sum(axis=1)
    
    # ADI = total days / non-zero days
    chunk_adi = np.divide(
        total_periods,
        chunk_non_zero_days,
        out=np.full(
            chunk_non_zero_days.shape,
            np.nan,
            dtype=np.float64
        ),
        where=chunk_non_zero_days > 0
    )
    
    chunk_non_zero_mean = np.divide(
        chunk_total_sales,
        chunk_non_zero_days,
        out=np.full(
            chunk_total_sales.shape,
            np.nan,
            dtype=np.float64
        ),
        where=chunk_non_zero_days > 0
    )
    
    # Population variance across observed non-zero demand values
    chunk_non_zero_variance = np.divide(
        chunk_square_sum,
        chunk_non_zero_days,
        out=np.full(
            chunk_square_sum.shape,
            np.nan,
            dtype=np.float64
        ),
        where=chunk_non_zero_days > 0
    ) - np.square(chunk_non_zero_mean)
    
    chunk_non_zero_variance = np.maximum(
        chunk_non_zero_variance,
        0
    )
    
    chunk_cv2 = np.divide(
        chunk_non_zero_variance,
        np.square(chunk_non_zero_mean),
        out=np.full(
            chunk_non_zero_variance.shape,
            np.nan,
            dtype=np.float64
        ),
        where=chunk_non_zero_mean > 0
    )
    
    adi_float64[start_row:end_row] = chunk_adi
    cv2_float64[start_row:end_row] = chunk_cv2

print("Float64 calculation complete.")

Float64 calculation complete.


In [16]:
demand_classification["ADI"] = adi_float64
demand_classification["CV2"] = cv2_float64

audit_conditions = [
    (
        demand_classification["ADI"].le(1.32)
        & demand_classification["CV2"].le(0.49)
    ),
    (
        demand_classification["ADI"].le(1.32)
        & demand_classification["CV2"].gt(0.49)
    ),
    (
        demand_classification["ADI"].gt(1.32)
        & demand_classification["CV2"].le(0.49)
    ),
    (
        demand_classification["ADI"].gt(1.32)
        & demand_classification["CV2"].gt(0.49)
    )
]

demand_classification["demand_type"] = np.select(
    audit_conditions,
    ["smooth", "erratic", "intermittent", "lumpy"],
    default="unclassified"
)

changed_classifications = (
    original_demand_types
    != demand_classification["demand_type"].reset_index(drop=True)
).sum()

print(
    "Changed demand classifications after float64 audit:",
    changed_classifications
)

print(
    demand_classification["demand_type"]
    .value_counts()
)

Changed demand classifications after float64 audit: 0
demand_type
intermittent    23075
lumpy            5938
smooth            983
erratic           494
Name: count, dtype: int64


In [17]:
demand_type_summary = (
    demand_classification
    .groupby("demand_type")
    .agg(
        series_count=("id", "count"),
        mean_ADI=("ADI", "mean"),
        mean_CV2=("CV2", "mean"),
        mean_zero_rate=("zero_rate", "mean"),
        mean_total_sales=("total_sales", "mean"),
        mean_non_zero_days=("non_zero_days", "mean")
    )
)

demand_type_summary["percentage"] = (
    demand_type_summary["series_count"]
    / len(demand_classification)
    * 100
)

demand_type_summary = demand_type_summary[
    [
        "series_count",
        "percentage",
        "mean_ADI",
        "mean_CV2",
        "mean_zero_rate",
        "mean_total_sales",
        "mean_non_zero_days"
    ]
].round(3)

demand_type_summary

,series_count,percentage,mean_ADI,mean_CV2,mean_zero_rate,mean_total_sales,mean_non_zero_days
demand_type,,,,,,,
erratic,494,1.620,1.205,0.724,16.686,13261.196,1617.119
intermittent,23075,75.681,7.259,0.287,73.458,1230.333,515.177
lumpy,5938,19.475,4.005,0.765,60.020,3125.854,776.015
smooth,983,3.224,1.168,0.361,13.798,13657.063,1673.177


In [18]:
demand_classification.to_csv(
    processed_data_path / "demand_classification.csv",
    index=False
)

demand_type_summary.to_csv(
    processed_data_path / "demand_type_summary.csv"
)

print("Final audited Step 11 files saved.")

Final audited Step 11 files saved.


## Step 11 Interpretation

ADI and CV² were calculated for all 30,490 item-store series. All 30,490 series produced finite ADI and CV² values, and 0 series were left unclassified.

Intermittent demand was the largest group, containing 23,075 series, or 75.681% of the dataset. Lumpy demand contained 5,938 series, or 19.475%. Together, intermittent and lumpy demand accounted for 29,013 series, equal to 95.156% of all item-store series.

Smooth demand contained 983 series, or 3.224%, while erratic demand contained 494 series, or 1.620%. The two demand types with frequent demand therefore represented only 1,477 series, equal to 4.844% of the dataset.

The mean ADI was 1.168 for smooth demand and 1.205 for erratic demand. In comparison, mean ADI increased to 7.259 for intermittent demand and 4.005 for lumpy demand. This means that non-zero demand occurred approximately once every 7.259 days for the average intermittent series and once every 4.005 days for the average lumpy series.

The mean CV² was 0.361 for smooth demand and 0.287 for intermittent demand, both below the classification threshold of 0.49. Mean CV² increased to 0.724 for erratic demand and 0.765 for lumpy demand, showing greater variation in the size of non-zero demand.

FOODS represented 72.63% of smooth series, 73.89% of erratic series, and 69.18% of lumpy series. Intermittent demand had a different category mix, with HOUSEHOLD representing 40.60%, FOODS representing 39.80%, and HOBBIES representing 19.60%.

The store distribution was also uneven. CA_3 contained 22.89% of all smooth series and 18.02% of all erratic series, while CA_4 contained 6.71% of smooth series and 4.45% of erratic series. The Step 12 subset must therefore preserve both category and store diversity rather than using an unstructured random sample.

There were 0 series with fewer than 10 non-zero sales days. The highest ADI was 161.750, corresponding to 12 non-zero demand days across the 1,941-day observation period. Therefore, the minimum activity rule will not exclude any series during subset selection.


### Surprise Log

The main surprise in Step 11 was that 29,013 of the 30,490 item-store series, equal to 95.156%, were classified as intermittent or lumpy. This is higher than the earlier result showing that 78.23% of series had more than 50% zero-sales days.

The difference occurs because an ADI above 1.32 corresponds to a zero-sales rate above approximately 24.24%, while the earlier sparsity measure used a stricter threshold of more than 50% zeros. The two findings measure different levels of sparsity and do not contradict each other.


# Step 12: Stratified Modelling Subset Selection

The complete dataset contains 30,490 item-store series, but the demand types are highly imbalanced. Intermittent demand accounts for 75.681% of all series, while erratic demand accounts for only 1.620%.

For balanced model comparison, I select 200 series from each demand type, giving a final modelling subset of 800 series.

Sampling is stratified by both category and store within each demand type. This prevents the subset from being dominated by one product category or store.

The subset is designed for comparison across demand types. It is not intended to reproduce the original population proportions.

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

np.random.seed(42)

processed_data_path = Path("../data/processed")
subset_data_path = Path("../data/subset")
merged_store_path = processed_data_path / "merged_by_store"

subset_data_path.mkdir(parents=True, exist_ok=True)

demand_classification = pd.read_csv(
    processed_data_path / "demand_classification.csv"
)

print("Classification shape:", demand_classification.shape)
print()
print(demand_classification["demand_type"].value_counts())

Classification shape: (30490, 12)

demand_type
intermittent    23075
lumpy            5938
smooth            983
erratic           494
Name: count, dtype: int64


eligible_series = demand_classification.loc[
    demand_classification["non_zero_days"] >= 10
].copy()

excluded_series = demand_classification.loc[
    demand_classification["non_zero_days"] < 10
].copy()

print("Total classified series:", len(demand_classification))
print("Eligible series:", len(eligible_series))
print("Excluded series:", len(excluded_series))
print(
    "Eligibility row match:",
    len(eligible_series) + len(excluded_series)
    == len(demand_classification)
)

In [23]:
eligible_series = demand_classification.loc[
    demand_classification["non_zero_days"] >= 10
].copy()

excluded_series = demand_classification.loc[
    demand_classification["non_zero_days"] < 10
].copy()

print("Total classified series:", len(demand_classification))
print("Eligible series:", len(eligible_series))
print("Excluded series:", len(excluded_series))
print(
    "Eligibility row match:",
    len(eligible_series) + len(excluded_series)
    == len(demand_classification)
)

Total classified series: 30490
Eligible series: 30490
Excluded series: 0
Eligibility row match: True


In [24]:
eligible_series["sampling_stratum"] = (
    eligible_series["cat_id"].astype(str)
    + "__"
    + eligible_series["store_id"].astype(str)
)

print(
    "Number of category-store strata:",
    eligible_series["sampling_stratum"].nunique()
)

eligible_series[
    [
        "id",
        "demand_type",
        "cat_id",
        "store_id",
        "sampling_stratum"
    ]
].head()

Number of category-store strata: 30


,id,demand_type,cat_id,store_id,sampling_stratum
0,HOBBIES_1_001_CA_1_evaluation,intermittent,HOBBIES,CA_1,HOBBIES__CA_1
1,HOBBIES_1_002_CA_1_evaluation,intermittent,HOBBIES,CA_1,HOBBIES__CA_1
2,HOBBIES_1_003_CA_1_evaluation,intermittent,HOBBIES,CA_1,HOBBIES__CA_1
3,HOBBIES_1_004_CA_1_evaluation,lumpy,HOBBIES,CA_1,HOBBIES__CA_1
4,HOBBIES_1_005_CA_1_evaluation,intermittent,HOBBIES,CA_1,HOBBIES__CA_1


In [26]:
def allocate_proportional_quotas(group_counts, target):
    """
    Allocate an exact sample target across available strata.
    """
    
    counts = group_counts.sort_index().astype(int)
    
    if target > counts.sum():
        raise ValueError(
            f"Target {target} exceeds available series {counts.sum()}."
        )
    
    quotas = pd.Series(
        0,
        index=counts.index,
        dtype=int
    )
    
    # Keep every available stratum represented where possible
    if target >= len(counts):
        quotas.loc[:] = 1
    
    remaining = target - int(quotas.sum())
    
    while remaining > 0:
        capacity = counts - quotas
        eligible_capacity = capacity[capacity > 0]
        
        if eligible_capacity.empty:
            raise ValueError(
                "No remaining capacity, but target has not been reached."
            )
        
        weights = (
            counts.loc[eligible_capacity.index]
            / counts.loc[eligible_capacity.index].sum()
        )
        
        exact_addition = remaining * weights
        floor_addition = np.floor(exact_addition).astype(int)
        
        floor_addition = pd.Series(
            np.minimum(
                floor_addition.to_numpy(),
                eligible_capacity.to_numpy()
            ),
            index=eligible_capacity.index,
            dtype=int
        )
        
        if floor_addition.sum() > 0:
            quotas.loc[floor_addition.index] += floor_addition
            remaining -= int(floor_addition.sum())
        
        if remaining == 0:
            break
        
        capacity = counts - quotas
        eligible_capacity = capacity[capacity > 0]
        
        fractional_remainders = (
            exact_addition - np.floor(exact_addition)
        ).reindex(
            eligible_capacity.index
        ).fillna(0)
        
        ordered_strata = sorted(
            eligible_capacity.index,
            key=lambda stratum: (
                fractional_remainders.get(stratum, 0),
                counts.loc[stratum]
            ),
            reverse=True
        )
        
        for stratum in ordered_strata:
            if remaining == 0:
                break
            
            if quotas.loc[stratum] < counts.loc[stratum]:
                quotas.loc[stratum] += 1
                remaining -= 1
    
    if int(quotas.sum()) != target:
        raise ValueError("Allocated quota does not match target.")
    
    if (quotas > counts).any():
        raise ValueError("A quota exceeds the available stratum size.")
    
    return quotas

In [27]:
target_per_demand_type = 200

demand_type_order = [
    "smooth",
    "erratic",
    "intermittent",
    "lumpy"
]

rng = np.random.default_rng(42)

selected_parts = []
quota_records = []

for demand_type in demand_type_order:
    type_data = eligible_series.loc[
        eligible_series["demand_type"] == demand_type
    ].copy()
    
    stratum_counts = (
        type_data["sampling_stratum"]
        .value_counts()
        .sort_index()
    )
    
    quotas = allocate_proportional_quotas(
        stratum_counts,
        target_per_demand_type
    )
    
    print(
        f"{demand_type}: selecting "
        f"{int(quotas.sum())} from {len(type_data)} series"
    )
    
    for stratum, sample_size in quotas.items():
        stratum_indices = type_data.index[
            type_data["sampling_stratum"] == stratum
        ].to_numpy()
        
        chosen_indices = rng.choice(
            stratum_indices,
            size=int(sample_size),
            replace=False
        )
        
        selected_parts.append(
            type_data.loc[chosen_indices].copy()
        )
        
        quota_records.append({
            "demand_type": demand_type,
            "sampling_stratum": stratum,
            "available_series": len(stratum_indices),
            "selected_series": int(sample_size)
        })

selected_series = pd.concat(
    selected_parts,
    ignore_index=True
)

sampling_quotas = pd.DataFrame(quota_records)

print()
print("Selected series:", len(selected_series))

smooth: selecting 200 from 983 series
erratic: selecting 200 from 494 series
intermittent: selecting 200 from 23075 series
lumpy: selecting 200 from 5938 series

Selected series: 800


In [28]:
selected_type_counts = (
    selected_series["demand_type"]
    .value_counts()
    .reindex(demand_type_order)
)

print("Selected series:", len(selected_series))
print("Unique selected IDs:", selected_series["id"].nunique())
print("Duplicate IDs:", selected_series["id"].duplicated().sum())
print()
print("Selected counts by demand type:")
print(selected_type_counts)

print()
print(
    "Exactly 200 per demand type:",
    selected_type_counts.eq(200).all()
)

print(
    "All selected series have at least 10 non-zero days:",
    selected_series["non_zero_days"].ge(10).all()
)

print(
    "All four demand types represented:",
    selected_series["demand_type"].nunique() == 4
)

Selected series: 800
Unique selected IDs: 800
Duplicate IDs: 0

Selected counts by demand type:
demand_type
smooth          200
erratic         200
intermittent    200
lumpy           200
Name: count, dtype: int64

Exactly 200 per demand type: True
All selected series have at least 10 non-zero days: True
All four demand types represented: True


In [29]:
population_counts = (
    eligible_series["demand_type"]
    .value_counts()
    .rename("population_series")
)

sample_counts = (
    selected_series["demand_type"]
    .value_counts()
    .rename("selected_series")
)

sampling_summary = pd.concat(
    [population_counts, sample_counts],
    axis=1
).fillna(0)

sampling_summary["population_percentage"] = (
    sampling_summary["population_series"]
    / len(eligible_series)
    * 100
)

sampling_summary["subset_percentage"] = (
    sampling_summary["selected_series"]
    / len(selected_series)
    * 100
)

sampling_summary["sampling_fraction"] = (
    sampling_summary["selected_series"]
    / sampling_summary["population_series"]
    * 100
)

sampling_summary = sampling_summary.loc[
    demand_type_order
].round(3)

sampling_summary

,population_series,selected_series,population_percentage,subset_percentage,sampling_fraction
demand_type,,,,,
smooth,983,200,3.224,25.0,20.346
erratic,494,200,1.620,25.0,40.486
intermittent,23075,200,75.681,25.0,0.867
lumpy,5938,200,19.475,25.0,3.368


In [31]:
selected_category_counts = pd.crosstab(
    selected_series["demand_type"],
    selected_series["cat_id"],
    margins=True
)

selected_category_counts

cat_id,FOODS,HOBBIES,HOUSEHOLD,All
demand_type,,,,
erratic,141,20,39,200
intermittent,80,42,78,200
lumpy,128,40,32,200
smooth,134,14,52,200
All,483,116,201,800


In [32]:
selected_category_percentage = (
    pd.crosstab(
        selected_series["demand_type"],
        selected_series["cat_id"],
        normalize="index"
    )
    * 100
).round(2)

selected_category_percentage

cat_id,FOODS,HOBBIES,HOUSEHOLD
demand_type,,,
erratic,70.5,10.0,19.5
intermittent,40.0,21.0,39.0
lumpy,64.0,20.0,16.0
smooth,67.0,7.0,26.0


In [33]:
population_category_percentage = (
    pd.crosstab(
        eligible_series["demand_type"],
        eligible_series["cat_id"],
        normalize="index"
    )
    * 100
).round(2)

category_balance_check = pd.concat(
    {
        "population": population_category_percentage,
        "selected": selected_category_percentage
    },
    axis=1
)

category_balance_check

population                   selected                  
cat_id            FOODS HOBBIES HOUSEHOLD    FOODS HOBBIES HOUSEHOLD
demand_type                                                         
erratic           73.89    8.10     18.02     70.5    10.0      19.5
intermittent      39.80   19.60     40.60     40.0    21.0      39.0
lumpy             69.18   17.82     13.00     64.0    20.0      16.0
smooth            72.63    2.95     24.42     67.0     7.0      26.0

In [34]:
selected_store_counts = pd.crosstab(
    selected_series["demand_type"],
    selected_series["store_id"],
    margins=True
)

selected_store_counts

store_id,CA_1,CA_2,CA_3,CA_4,TX_1,TX_2,TX_3,WI_1,WI_2,WI_3,All
demand_type,,,,,,,,,,,
erratic,18,21,34,10,22,24,19,17,15,20,200
intermittent,20,19,18,22,20,19,20,21,21,20,200
lumpy,20,24,23,15,20,23,19,16,20,20,200
smooth,27,16,42,15,18,22,19,14,12,15,200
All,85,80,117,62,80,88,77,68,68,75,800


In [35]:
selected_store_percentage = (
    pd.crosstab(
        selected_series["demand_type"],
        selected_series["store_id"],
        normalize="index"
    )
    * 100
).round(2)

selected_store_percentage

store_id,CA_1,CA_2,CA_3,CA_4,TX_1,TX_2,TX_3,WI_1,WI_2,WI_3
demand_type,,,,,,,,,,
erratic,9.0,10.5,17.0,5.0,11.0,12.0,9.5,8.5,7.5,10.0
intermittent,10.0,9.5,9.0,11.0,10.0,9.5,10.0,10.5,10.5,10.0
lumpy,10.0,12.0,11.5,7.5,10.0,11.5,9.5,8.0,10.0,10.0
smooth,13.5,8.0,21.0,7.5,9.0,11.0,9.5,7.0,6.0,7.5


In [36]:
population_store_percentage = (
    pd.crosstab(
        eligible_series["demand_type"],
        eligible_series["store_id"],
        normalize="index"
    )
    * 100
).round(2)

store_balance_check = pd.concat(
    {
        "population": population_store_percentage,
        "selected": selected_store_percentage
    },
    axis=1
)

store_balance_check

population                                                   \
store_id           CA_1   CA_2   CA_3   CA_4   TX_1   TX_2   TX_3   WI_1   
demand_type                                                                
erratic            8.50  10.12  18.02   4.45  10.93  12.35   9.51   8.50   
intermittent       9.95   9.46   8.84  11.04  10.04   9.47  10.15  10.73   
lumpy              9.73  12.50  11.70   6.96   9.94  11.70   9.53   7.92   
smooth            13.63   7.43  22.89   6.71   9.05  10.89   9.46   6.21   

                           selected                                            \
store_id       WI_2   WI_3     CA_1  CA_2  CA_3  CA_4  TX_1  TX_2  TX_3  WI_1   
demand_type                                                                     
erratic        7.29  10.32      9.0  10.5  17.0   5.0  11.0  12.0   9.5   8.5   
intermittent  10.24  10.08     10.0   9.5   9.0  11.0  10.0   9.5  10.0  10.5   
lumpy          9.90  10.12     10.0  12.0  11.5   7.5  10.0  11.5   9.5   8.0   
smooth         6.41   7.32     13.5   8.0  21.0   7.5   9.0  11.0   9.5   7.0   

                          
store_id      WI_2  WI_3  
demand_type               
erratic        7.5  10.0  
intermittent  10.5  10.0  
lumpy         10.0  10.0  
smooth         6.0   7.5

In [37]:
selected_series_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "ADI",
    "CV2",
    "demand_type",
    "zero_rate",
    "total_sales",
    "non_zero_days",
    "sampling_stratum"
]

selected_series = selected_series[
    selected_series_columns
].copy()

selected_series["demand_type"] = pd.Categorical(
    selected_series["demand_type"],
    categories=demand_type_order,
    ordered=True
)

selected_series = selected_series.sort_values(
    ["demand_type", "store_id", "cat_id", "id"]
).reset_index(drop=True)

selected_series.to_csv(
    subset_data_path / "selected_series.csv",
    index=False
)

sampling_quotas.to_csv(
    subset_data_path / "sampling_quotas.csv",
    index=False
)

sampling_summary.to_csv(
    subset_data_path / "sampling_summary.csv"
)

category_balance_check.to_csv(
    subset_data_path / "category_balance_check.csv"
)

store_balance_check.to_csv(
    subset_data_path / "store_balance_check.csv"
)

print("Selected-series files saved.")

Selected-series files saved.


In [39]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

expected_subset_rows = (
    len(selected_series) * 1941
)

print("Expected subset rows:", expected_subset_rows)

Expected subset rows: 1552800


In [40]:
subset_parts = []
extraction_records = []

selected_store_ids = sorted(
    selected_series["store_id"].unique()
)

for store in selected_store_ids:
    selected_ids_for_store = (
        selected_series.loc[
            selected_series["store_id"] == store,
            "id"
        ]
        .astype(str)
        .tolist()
    )
    
    source_file = (
        merged_store_path
        / f"{store}_merged.parquet"
    )
    
    parquet_file = pq.ParquetFile(source_file)
    
    selected_id_array = pa.array(
        selected_ids_for_store
    )
    
    store_filtered_parts = []
    
    for batch in parquet_file.iter_batches(
        batch_size=250_000
    ):
        id_column_position = (
            batch.schema.get_field_index("id")
        )
        
        id_column = batch.column(
            id_column_position
        )
        
        selection_mask = pc.is_in(
            id_column,
            value_set=selected_id_array
        )
        
        filtered_batch = batch.filter(
            selection_mask
        )
        
        if filtered_batch.num_rows > 0:
            store_filtered_parts.append(
                filtered_batch.to_pandas()
            )
    
    store_subset = pd.concat(
        store_filtered_parts,
        ignore_index=True
    )
    
    expected_store_rows = (
        len(selected_ids_for_store) * 1941
    )
    
    actual_store_rows = len(store_subset)
    
    extraction_records.append({
        "store_id": store,
        "selected_series": len(selected_ids_for_store),
        "expected_rows": expected_store_rows,
        "actual_rows": actual_store_rows,
        "row_match": expected_store_rows == actual_store_rows
    })
    
    print(
        f"{store}: {len(selected_ids_for_store)} series, "
        f"{actual_store_rows:,} rows"
    )
    
    subset_parts.append(store_subset)
    
    del store_filtered_parts
    del store_subset
    gc.collect()

subset_extraction_validation = pd.DataFrame(
    extraction_records
)

subset_extraction_validation

CA_1: 85 series, 164,985 rows
CA_2: 80 series, 155,280 rows
CA_3: 117 series, 227,097 rows
CA_4: 62 series, 120,342 rows
TX_1: 80 series, 155,280 rows
TX_2: 88 series, 170,808 rows
TX_3: 77 series, 149,457 rows
WI_1: 68 series, 131,988 rows
WI_2: 68 series, 131,988 rows
WI_3: 75 series, 145,575 rows


,store_id,selected_series,expected_rows,actual_rows,row_match
0,CA_1,85,164985,164985,True
1,CA_2,80,155280,155280,True
2,CA_3,117,227097,227097,True
3,CA_4,62,120342,120342,True
4,TX_1,80,155280,155280,True
5,TX_2,88,170808,170808,True
6,TX_3,77,149457,149457,True
7,WI_1,68,131988,131988,True
8,WI_2,68,131988,131988,True
9,WI_3,75,145575,145575,True


In [41]:
selected_m5_data = pd.concat(
    subset_parts,
    ignore_index=True
)

selected_metadata = selected_series[
    [
        "id",
        "demand_type",
        "ADI",
        "CV2",
        "zero_rate"
    ]
].copy()

selected_m5_data = selected_m5_data.merge(
    selected_metadata,
    on="id",
    how="left",
    validate="many_to_one"
)

print("Selected M5 data shape:", selected_m5_data.shape)

Selected M5 data shape: (1552800, 24)


In [42]:
rows_per_series = (
    selected_m5_data
    .groupby("id")
    .size()
)

subset_sales_totals = (
    selected_m5_data
    .groupby("id", as_index=False)["sales"]
    .sum()
    .rename(columns={"sales": "subset_total_sales"})
)

sales_total_validation = selected_series[
    ["id", "total_sales"]
].merge(
    subset_sales_totals,
    on="id",
    how="left",
    validate="one_to_one"
)

sales_total_validation["sales_match"] = (
    sales_total_validation["total_sales"]
    == sales_total_validation["subset_total_sales"]
)

print("Expected subset rows:", expected_subset_rows)
print("Actual subset rows:", len(selected_m5_data))
print(
    "Subset row match:",
    len(selected_m5_data) == expected_subset_rows
)

print(
    "Unique selected series:",
    selected_m5_data["id"].nunique()
)

print(
    "Minimum rows per series:",
    rows_per_series.min()
)

print(
    "Maximum rows per series:",
    rows_per_series.max()
)

print(
    "All extraction row checks valid:",
    subset_extraction_validation["row_match"].all()
)

print(
    "All selected sales totals match:",
    sales_total_validation["sales_match"].all()
)

print(
    "Missing demand types:",
    selected_m5_data["demand_type"].isna().sum()
)

Expected subset rows: 1552800
Actual subset rows: 1552800
Subset row match: True
Unique selected series: 800
Minimum rows per series: 1941
Maximum rows per series: 1941
All extraction row checks valid: True
All selected sales totals match: True
Missing demand types: 0


In [43]:
selected_m5_data = selected_m5_data.sort_values(
    ["demand_type", "store_id", "id", "date"]
).reset_index(drop=True)

selected_m5_data.to_parquet(
    subset_data_path / "selected_m5_data.parquet",
    index=False,
    engine="pyarrow",
    compression="zstd"
)

subset_extraction_validation.to_csv(
    subset_data_path / "subset_extraction_validation.csv",
    index=False
)

sales_total_validation.to_csv(
    subset_data_path / "subset_sales_total_validation.csv",
    index=False
)

print(
    "Saved:",
    subset_data_path / "selected_m5_data.parquet"
)

Saved: ..\data\subset\selected_m5_data.parquet


In [44]:
saved_subset_file = (
    subset_data_path / "selected_m5_data.parquet"
)

saved_subset_metadata = pq.ParquetFile(
    saved_subset_file
)

print(
    "Saved parquet rows:",
    saved_subset_metadata.metadata.num_rows
)

print(
    "Saved parquet row match:",
    saved_subset_metadata.metadata.num_rows
    == expected_subset_rows
)

Saved parquet rows: 1552800
Saved parquet row match: True


## Step 12 Interpretation

A balanced modelling subset of 800 item-store series was selected from the 30,490 classified series. The subset contains 200 smooth, 200 erratic, 200 intermittent, and 200 lumpy series, meaning that each demand type represents 25.0% of the modelling sample.

The sampling fractions differ because the original demand groups are highly imbalanced. The subset contains 40.486% of all erratic series, 20.346% of all smooth series, 3.368% of all lumpy series, and 0.867% of all intermittent series. Equal sampling was used to provide 200 observations for each demand type rather than reproduce the original population distribution.

The category composition was broadly preserved within each demand type. For intermittent demand, FOODS changed from 39.80% in the population to 40.0% in the subset, HOBBIES changed from 19.60% to 21.0%, and HOUSEHOLD changed from 40.60% to 39.0%. The largest category difference occurred for smooth FOODS, which decreased from 72.63% to 67.0%, a difference of 5.63 percentage points.

The store distribution was also preserved. CA_3 remained the largest contributor to smooth demand, accounting for 22.89% of the population and 21.0% of the selected smooth series. For intermittent demand, every store contributed between 18 and 22 series, equal to between 9.0% and 11.0% of the selected group.

The final modelling dataset contains 1,552,800 item-store-day rows, calculated from 800 series multiplied by 1,941 days. Every selected series contains exactly 1,941 rows, all 800 sales totals match the original classification data, and 0 demand-type values are missing.

Because the subset is balanced while the original population is imbalanced, model comparisons will be reported separately for each demand type. An unweighted overall score from the subset will not be treated as a population-level M5 result.


### Surprise Log

The main surprise in Step 12 was the difference in sampling fractions required to create a balanced subset. Selecting 200 erratic series used 40.486% of the available erratic population, while selecting 200 intermittent series used only 0.867% of the available intermittent population.

This shows how strongly the full dataset is dominated by intermittent demand. The balanced subset is useful for fair model comparison across four demand types, but its overall distribution does not represent the original population.


In [45]:
from pathlib import Path
import pandas as pd

project_root = Path("..")
figures_path = project_root / "outputs" / "figures"
tables_path = project_root / "outputs" / "tables"
model_results_path = project_root / "outputs" / "model_results"

for folder in [figures_path, tables_path, model_results_path]:
    folder.mkdir(parents=True, exist_ok=True)

def show_saved_files(folder):
    files = [
        file for file in folder.rglob("*")
        if file.is_file()
    ]
    
    print(f"\nFolder: {folder}")
    print(f"Number of files: {len(files)}")
    
    for file in sorted(files):
        print(
            f"- {file.name} "
            f"({file.stat().st_size / 1024:.1f} KB)"
        )

show_saved_files(figures_path)
show_saved_files(tables_path)
show_saved_files(model_results_path)


Folder: ..\outputs\figures
Number of files: 7
- category_sales_rolling_28.png (273.2 KB)
- event_vs_non_event_sales.png (87.1 KB)
- snap_vs_non_snap_sales_by_state.png (100.7 KB)
- state_sales_rolling_28.png (341.3 KB)
- store_sales_rolling_28.png (810.6 KB)
- total_daily_sales_rolling_28.png (414.5 KB)
- weekly_total_sales.png (241.9 KB)

Folder: ..\outputs\tables
Number of files: 25
- average_sales_by_month.csv (0.3 KB)
- average_sales_by_weekday.csv (0.2 KB)
- category_daily_sales.csv (160.1 KB)
- category_growth_summary.csv (0.3 KB)
- category_price_summary.csv (0.2 KB)
- category_price_variability.csv (0.2 KB)
- category_sales_summary.csv (0.1 KB)
- daily_total_sales.csv (129.4 KB)
- department_price_summary.csv (0.4 KB)
- department_sales_summary.csv (0.1 KB)
- event_name_sales_summary.csv (1.0 KB)
- event_sales_summary.csv (0.2 KB)
- event_type_sales_summary.csv (0.3 KB)
- monthly_total_sales.csv (1.3 KB)
- price_change_by_category.csv (0.2 KB)
- price_change_by_department.csv 

In [46]:
from datetime import datetime

output_records = []

for output_type, folder in {
    "figure": figures_path,
    "table": tables_path,
    "model_result": model_results_path
}.items():
    
    for file in folder.rglob("*"):
        if file.is_file():
            output_records.append({
                "output_type": output_type,
                "filename": file.name,
                "relative_path": str(file.relative_to(project_root)),
                "file_extension": file.suffix.lower(),
                "size_kb": round(file.stat().st_size / 1024, 2),
                "last_modified": datetime.fromtimestamp(
                    file.stat().st_mtime
                )
            })

output_manifest = pd.DataFrame(output_records)

if len(output_manifest) > 0:
    output_manifest = output_manifest.sort_values(
        ["output_type", "filename"]
    ).reset_index(drop=True)

output_manifest

,output_type,filename,relative_path,file_extension,size_kb,last_modified
0,figure,category_sales_rolling_28.png,outputs\figures\category_sales_rolling_28.png,.png,273.17,2026-07-03 21:29:49.930131
1,figure,event_vs_non_event_sales.png,outputs\figures\event_vs_non_event_sales.png,.png,87.09,2026-07-04 14:11:59.816958
2,figure,snap_vs_non_snap_sales_by_state.png,outputs\figures\snap_vs_non_snap_sales_by_stat...,.png,100.73,2026-07-04 14:12:42.444023
3,figure,state_sales_rolling_28.png,outputs\figures\state_sales_rolling_28.png,.png,341.29,2026-07-03 21:29:58.844984
4,figure,store_sales_rolling_28.png,outputs\figures\store_sales_rolling_28.png,.png,810.59,2026-07-03 21:30:08.825349
5,figure,total_daily_sales_rolling_28.png,outputs\figures\total_daily_sales_rolling_28.png,.png,414.51,2026-07-03 21:22:08.106452
6,figure,weekly_total_sales.png,outputs\figures\weekly_total_sales.png,.png,241.87,2026-07-03 21:22:14.702509
7,table,average_sales_by_month.csv,outputs\tables\average_sales_by_month.csv,.csv,0.26,2026-07-03 21:22:28.263528
8,table,average_sales_by_weekday.csv,outputs\tables\average_sales_by_weekday.csv,.csv,0.21,2026-07-03 21:22:28.262494
9,table,category_daily_sales.csv,outputs\tables\category_daily_sales.csv,.csv,160.06,2026-07-03 21:29:42.739802


In [47]:
output_manifest.to_csv(
    tables_path / "output_file_manifest.csv",
    index=False
)

print(
    "Saved:",
    tables_path / "output_file_manifest.csv"
)

Saved: ..\outputs\tables\output_file_manifest.csv


In [48]:
required_files = [
    project_root / "data" / "processed" / "sales_long_store_row_counts.csv",
    project_root / "data" / "processed" / "step10_merge_validation.csv",
    project_root / "data" / "processed" / "m5_merged_base.parquet",
    project_root / "data" / "processed" / "demand_classification.csv",
    project_root / "data" / "processed" / "demand_type_summary.csv",
    project_root / "data" / "subset" / "selected_series.csv",
    project_root / "data" / "subset" / "selected_m5_data.parquet",
    project_root / "data" / "subset" / "sampling_summary.csv",
    project_root / "data" / "subset" / "category_balance_check.csv",
    project_root / "data" / "subset" / "store_balance_check.csv",
    project_root / "data" / "subset" / "subset_extraction_validation.csv",
    project_root / "data" / "subset" / "subset_sales_total_validation.csv"
]

file_check = pd.DataFrame({
    "file": [str(file) for file in required_files],
    "exists": [file.exists() for file in required_files],
    "size_mb": [
        round(file.stat().st_size / (1024 ** 2), 2)
        if file.exists() else None
        for file in required_files
    ]
})

file_check

,file,exists,size_mb
0,..\data\processed\sales_long_store_row_counts.csv,True,0.00
1,..\data\processed\step10_merge_validation.csv,True,0.00
2,..\data\processed\m5_merged_base.parquet,True,196.16
3,..\data\processed\demand_classification.csv,True,4.23
4,..\data\processed\demand_type_summary.csv,True,0.00
5,..\data\subset\selected_series.csv,True,0.12
6,..\data\subset\selected_m5_data.parquet,True,0.92
7,..\data\subset\sampling_summary.csv,True,0.00
8,..\data\subset\category_balance_check.csv,True,0.00
9,..\data\subset\store_balance_check.csv,True,0.00
